# Notebook 3: ML + Streaming Integration

This notebook combines:
- **Batch ML**: Pre-trained ALS model from `01_als_training.ipynb` (Top-5 recommendations per user)
- **Live Streaming**: Window analytics written by `02_streaming.ipynb` to `/data/streaming_output/`

Goal: Re-rank each user's ALS recommendations using the live trending score, and measure latency.

## 1. Start Spark Session

In [23]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('ML_Streaming_Integration') \
    .master('spark://spark-master:7077') \
    .config('spark.executor.memory', '1g') \
    .config('spark.driver.memory', '1g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.cores.max', '1') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

Spark version: 3.5.0


In [24]:
# Read raw without schema to see if files have any content at all
import subprocess
result = subprocess.run(
    ['find', '/data/streaming_output/', '-name', '*.parquet', '-size', '+0c'],
    capture_output=True, text=True
)
print('Non-empty parquet files:')
print(result.stdout if result.stdout else 'None — all files are empty (0 bytes)')

Non-empty parquet files:
/data/streaming_output/part-00000-01df6eb6-a1e2-4012-a057-e98689e73ae5-c000.snappy.parquet
/data/streaming_output/part-00000-034b3e12-23d6-4f21-b8ae-bff358890e60-c000.snappy.parquet
/data/streaming_output/part-00000-03a1e437-e8d6-4fb9-aabf-71896d2122c3-c000.snappy.parquet
/data/streaming_output/part-00000-04fea45b-2604-4189-86bf-1e14c1033c4f-c000.snappy.parquet
/data/streaming_output/part-00000-0549e525-dc97-417a-8a7f-e0654ec0dc96-c000.snappy.parquet
/data/streaming_output/part-00000-05e47063-46aa-4fc1-8939-fa0d68569824-c000.snappy.parquet
/data/streaming_output/part-00000-062f6d9e-a1d9-4606-93a8-c2491d183f8f-c000.snappy.parquet
/data/streaming_output/part-00000-0700448d-7340-4f78-9009-2189e3eb9389-c000.snappy.parquet
/data/streaming_output/part-00000-071bd537-5ca9-4d15-abe3-0df4888b4370-c000.snappy.parquet
/data/streaming_output/part-00000-08327802-5a5d-46b9-8443-d650a5301a8e-c000.snappy.parquet
/data/streaming_output/part-00000-089aac11-7e8a-47cf-8ffe-117749a

## 2. Load the Pre-Trained ALS Model

In [25]:
from pyspark.ml.recommendation import ALSModel

MODEL_PATH = '/data/als_model'

model = ALSModel.load(MODEL_PATH)
print(f'ALS model loaded from {MODEL_PATH}')
print(f'  Rank : {model.rank}')

ALS model loaded from /data/als_model
  Rank : 50


## 3. Load Pre-Generated Top-5 Recommendations (Batch)

In [26]:
from pyspark.sql import functions as F

RECS_PATH = '/data/user_top5_recs.parquet'

recs_df = spark.read.parquet(RECS_PATH)

print(f'Top-5 recommendations loaded from {RECS_PATH}')
print(f'Total rows      : {recs_df.count():,}')
print(f'Unique users    : {recs_df.select("user_id").distinct().count():,}')
recs_df.show(10)

Top-5 recommendations loaded from /data/user_top5_recs.parquet


Total rows      : 7,218,965


Unique users    : 1,443,793
+-------+-------+----------------+
|user_id|item_id|predicted_rating|
+-------+-------+----------------+
|     21| 412798|       17.308502|
|     21| 223332|       16.829039|
|     21|  74945|       16.443571|
|     21|  65366|       16.066801|
|     21|  51876|       15.901307|
|     29| 178947|        4.998671|
|     29| 216640|        4.826401|
|     29|  69861|         4.45771|
|     29| 229628|       4.4559135|
|     29| 354081|        4.450902|
+-------+-------+----------------+
only showing top 10 rows



## 4. Load Live Streaming Output (from Notebook 02)

In [27]:
import os

STREAM_OUTPUT_PATH = '/data/streaming_output/'

# Check that streaming output files exist
files = [f for f in os.listdir(STREAM_OUTPUT_PATH) if f.endswith('.parquet')] \
    if os.path.exists(STREAM_OUTPUT_PATH) else []

print(f'Streaming output path : {STREAM_OUTPUT_PATH}')
print(f'Parquet files found   : {len(files)}')

if len(files) == 0:
    print('')
    print('  WARNING: No streaming output found yet.')
    print('  Make sure 02_streaming.ipynb is running and kafka_producer.py is active.')
    print('  Re-run this cell after a few batches have been written (30–60 seconds).')
else:
    streaming_df = spark.read.parquet(STREAM_OUTPUT_PATH)
    print(f'Rows loaded           : {streaming_df.count():,}')
    streaming_df.show(10)

Streaming output path : /data/streaming_output/
Parquet files found   : 932


Rows loaded           : 99,728
+-------------------+-------------------+-------+----------+-----------------+--------------+
|       window_start|         window_end|item_id|avg_rating|interaction_count|trending_score|
+-------------------+-------------------+-------+----------+-----------------+--------------+
|2026-05-12 16:19:40|2026-05-12 16:20:10| 123352|       3.0|                1|           3.0|
|2026-05-12 16:24:20|2026-05-12 16:24:50| 405177|       1.0|                1|           1.0|
|2026-05-12 16:19:10|2026-05-12 16:19:40| 254691|       4.0|                1|           4.0|
|2026-05-12 16:19:30|2026-05-12 16:20:00| 241686|       3.0|                1|           3.0|
|2026-05-12 16:24:10|2026-05-12 16:24:40| 173483|       1.0|                1|           1.0|
|2026-05-12 16:18:20|2026-05-12 16:18:50| 303135|       5.0|                1|           5.0|
|2026-05-12 16:18:20|2026-05-12 16:18:50| 255541|       5.0|                1|           5.0|
|2026-05-12 16:26:40|2026-05-

## 5. Get the Latest Trending Score per Item

Multiple batches may have been written for the same item across different windows.
We keep only the **most recent window** result per item.

In [28]:
from pyspark.sql.window import Window

# Rank rows per item by most recent window_end, keep rank=1 (latest)
window_spec = Window.partitionBy('item_id').orderBy(F.col('window_end').desc())

latest_trending = (
    streaming_df
    .withColumn('rank', F.row_number().over(window_spec))
    .filter(F.col('rank') == 1)
    .select('item_id', 'avg_rating', 'interaction_count', 'trending_score', 'window_end')
)

print(f'Unique items with live trending data : {latest_trending.count():,}')
print('Top 10 trending items right now:')
latest_trending.orderBy(F.col('trending_score').desc()).show(10)

Unique items with live trending data : 19,989
Top 10 trending items right now:


[Stage 39:======================================================> (29 + 1) / 30]

+-------+----------+-----------------+--------------+-------------------+
|item_id|avg_rating|interaction_count|trending_score|         window_end|
+-------+----------+-----------------+--------------+-------------------+
| 348180|       5.0|                2|          10.0|2026-05-12 17:07:40|
| 108773|       5.0|                2|          10.0|2026-05-12 17:06:20|
| 120177|       5.0|                2|          10.0|2026-05-12 16:38:00|
| 201302|       5.0|                2|          10.0|2026-05-12 17:01:20|
| 217782|       5.0|                2|          10.0|2026-05-12 17:10:50|
| 349228|       5.0|                2|          10.0|2026-05-12 16:32:40|
| 377415|       5.0|                2|          10.0|2026-05-12 16:44:10|
| 381981|       5.0|                2|          10.0|2026-05-12 16:39:00|
| 394632|       5.0|                2|          10.0|2026-05-12 17:09:00|
| 409025|       5.0|                2|          10.0|2026-05-12 17:00:20|
+-------+----------+-----------------+

## 6. Re-Rank ALS Recommendations Using Trending Score

**Integration logic:**
- Join each user's Top-5 ALS recommendations with the live trending score of those items
- Compute a **blended score**: `0.7 × predicted_rating + 0.3 × normalized_trending_score`
- Re-rank the 5 recommendations by blended score
- Items with no live data keep their original ALS ranking (trending_score = 0)

In [29]:
# Normalize trending score to [0, 5] so it's on the same scale as ALS predicted_rating
max_trending = latest_trending.agg(F.max('trending_score')).collect()[0][0] or 1.0

trending_normalized = latest_trending.withColumn(
    'trending_norm',
    F.round((F.col('trending_score') / max_trending) * 5.0, 4)
)

# Join ALS recs with live trending data (left join: keep all recs even if no live data)
joined = recs_df.join(
    trending_normalized.select('item_id', 'trending_norm', 'trending_score'),
    on='item_id',
    how='left'
).fillna({'trending_norm': 0.0, 'trending_score': 0.0})

# Blended score: 70% ALS + 30% live trending
# Rationale: ALS captures long-term preference; trending captures real-time popularity
integrated = joined.withColumn(
    'blended_score',
    F.round(0.7 * F.col('predicted_rating') + 0.3 * F.col('trending_norm'), 4)
)

# Re-rank within each user by blended score
user_window = Window.partitionBy('user_id').orderBy(F.col('blended_score').desc())

final_recs = (
    integrated
    .withColumn('final_rank', F.row_number().over(user_window))
    .filter(F.col('final_rank') <= 5)
    .select(
        'user_id', 'item_id', 'predicted_rating',
        'trending_norm', 'trending_score', 'blended_score', 'final_rank'
    )
    .orderBy('user_id', 'final_rank')
)

print('Integration complete.')
print('  Formula: blended_score = 0.7 x predicted_rating + 0.3 x trending_norm')
print(f'  Max trending score seen : {max_trending:.4f}')
print(f'  Total recommendation rows: {final_recs.count():,}')
final_recs.show(20)

Integration complete.
  Formula: blended_score = 0.7 x predicted_rating + 0.3 x trending_norm
  Max trending score seen : 10.0000


  Total recommendation rows: 7,218,965


[Stage 62:===========================================>              (3 + 1) / 4]

+-------+-------+----------------+-------------+--------------+-------------+----------+
|user_id|item_id|predicted_rating|trending_norm|trending_score|blended_score|final_rank|
+-------+-------+----------------+-------------+--------------+-------------+----------+
|      0| 124636|       1.1231183|          2.5|           5.0|       1.5362|         1|
|      0| 144788|       1.1452341|          0.0|           0.0|       0.8017|         2|
|      0| 220479|        1.127475|          0.0|           0.0|       0.7892|         3|
|      0| 111545|       1.1255709|          0.0|           0.0|       0.7879|         4|
|      0| 138314|        1.125554|          0.0|           0.0|       0.7879|         5|
|      1| 165345|       3.9989884|          2.5|           5.0|       3.5493|         1|
|      1| 404963|       3.7396314|          0.0|           0.0|       2.6177|         2|
|      1|  62648|       3.6535776|          0.0|           0.0|       2.5575|         3|
|      1|  72210|    

## 7. Sample Output — Top-5 for Specific Users

In [30]:
import pickle

# Load label encoders so we can decode integer IDs back to original strings
with open('/data/label_encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)

item_encoder = encoders['item']

# Pick 3 sample users to display
sample_users = [
    row['user_id'] for row in final_recs.select('user_id').distinct().limit(3).collect()
]

print('=== Sample Recommendations (Batch ALS + Live Trending) ===\n')
for uid in sample_users:
    user_recs = final_recs.filter(F.col('user_id') == uid).collect()
    print(f'User {uid}:')
    for row in user_recs:
        try:
            item_name = item_encoder.inverse_transform([row['item_id']])[0]
        except Exception:
            item_name = str(row['item_id'])
        trending_flag = ' *** TRENDING ***' if row['trending_score'] > 0 else ''
        print(f'  Rank {row["final_rank"]} | item={item_name} '
              f'| ALS={row["predicted_rating"]:.2f} '
              f'| trending={row["trending_norm"]:.2f} '
              f'| blended={row["blended_score"]:.2f}{trending_flag}')
    print()

=== Sample Recommendations (Batch ALS + Live Trending) ===



User 9:
  Rank 1 | item=B00B5L35GE | ALS=0.79 | trending=2.50 | blended=1.30 *** TRENDING ***
  Rank 2 | item=B0C6MDB88N | ALS=0.77 | trending=2.50 | blended=1.29 *** TRENDING ***
  Rank 3 | item=B00PRRADBM | ALS=1.00 | trending=0.00 | blended=0.70
  Rank 4 | item=B00ARDJKWO | ALS=0.80 | trending=0.00 | blended=0.56
  Rank 5 | item=B07CWQGX93 | ALS=0.79 | trending=0.00 | blended=0.56



User 13:
  Rank 1 | item=B07XXZ2MMV | ALS=4.00 | trending=0.00 | blended=2.80
  Rank 2 | item=B08QH2TTMR | ALS=3.79 | trending=0.00 | blended=2.65
  Rank 3 | item=B01LYNSL5S | ALS=3.71 | trending=0.00 | blended=2.59
  Rank 4 | item=B07PMFD12W | ALS=3.48 | trending=0.00 | blended=2.44
  Rank 5 | item=B01CM85C0K | ALS=3.46 | trending=0.00 | blended=2.42



User 14:
  Rank 1 | item=B01LYHEC16 | ALS=5.17 | trending=2.50 | blended=4.37 *** TRENDING ***
  Rank 2 | item=B0991HKP47 | ALS=5.61 | trending=0.00 | blended=3.93
  Rank 3 | item=B00M7M4GLI | ALS=5.46 | trending=0.00 | blended=3.82
  Rank 4 | item=B00UXNZ1BM | ALS=5.20 | trending=0.00 | blended=3.64
  Rank 5 | item=B071G3D8QD | ALS=5.18 | trending=0.00 | blended=3.63



## 8. Latency Measurement

Measure end-to-end latency: from the time a streaming event arrives to when a recommendation is updated.
Target from the project spec: **< 5 seconds** (bonus point).

In [31]:
import time
from datetime import datetime, timezone

# Simulate end-to-end latency:
# t0 = now (a new streaming event just arrived)
# t1 = after loading latest streaming_output + re-ranking recommendations

t0 = time.time()

# --- Pipeline: read latest streaming output ---
fresh_streaming = spark.read.parquet(STREAM_OUTPUT_PATH)

# --- Get latest trending per item ---
ws = Window.partitionBy('item_id').orderBy(F.col('window_end').desc())
fresh_trending = (
    fresh_streaming
    .withColumn('rank', F.row_number().over(ws))
    .filter(F.col('rank') == 1)
    .select('item_id', 'trending_score')
)

max_t = fresh_trending.agg(F.max('trending_score')).collect()[0][0] or 1.0
fresh_trending = fresh_trending.withColumn(
    'trending_norm', F.round((F.col('trending_score') / max_t) * 5.0, 4)
)

# --- Re-rank for a single incoming user ---
test_user_id = sample_users[0]

user_recs_df = recs_df.filter(F.col('user_id') == test_user_id)

updated = (
    user_recs_df
    .join(fresh_trending, on='item_id', how='left')
    .fillna({'trending_norm': 0.0, 'trending_score': 0.0})
    .withColumn(
        'blended_score',
        F.round(0.7 * F.col('predicted_rating') + 0.3 * F.col('trending_norm'), 4)
    )
    .orderBy(F.col('blended_score').desc())
    .limit(5)
)

# Force computation (action)
result_count = updated.count()

t1 = time.time()
latency = t1 - t0

print(f'=== Latency Measurement ===')
print(f'User tested          : {test_user_id}')
print(f'Recommendations      : {result_count}')
print(f'End-to-end latency   : {latency:.3f} seconds')
print()
if latency < 5.0:
    print(f'  PASS — latency {latency:.3f}s < 5s target (bonus point achieved)')
else:
    print(f'  INFO — latency {latency:.3f}s > 5s target')
    print(f'  Tip: pre-cache recs_df with recs_df.cache() to reduce re-read time.')

[Stage 112:============================>                            (1 + 1) / 2]

=== Latency Measurement ===
User tested          : 9
Recommendations      : 5
End-to-end latency   : 201.053 seconds

  INFO — latency 201.053s > 5s target
  Tip: pre-cache recs_df with recs_df.cache() to reduce re-read time.


## 9. Save Final Integrated Recommendations

In [32]:
OUTPUT_PATH = '/data/integrated_recs.parquet'

final_recs.write.mode('overwrite').parquet(OUTPUT_PATH)

print(f'Integrated recommendations saved → {OUTPUT_PATH}')
print(f'Total rows : {final_recs.count():,}')
print()
print('This file is read by dashboard/app.py to display live recommendations.')

Integrated recommendations saved → /data/integrated_recs.parquet


[Stage 130:============================>                            (1 + 1) / 2]

Total rows : 7,218,965

This file is read by dashboard/app.py to display live recommendations.


## 10. Integration Summary

In [33]:
print('=== Integration Summary ===')
print()
print('Data sources:')
print(f'  Batch  — ALS model          : {MODEL_PATH}')
print(f'  Batch  — Pre-gen Top-5 recs : {RECS_PATH}')
print(f'  Stream — Window analytics   : {STREAM_OUTPUT_PATH}')
print()
print('Integration logic:')
print('  1. Load ALS Top-5 per user (historical preference)')
print('  2. Load latest trending score per item (real-time signal)')
print('  3. Normalize trending score to [0, 5] (same scale as ALS rating)')
print('  4. Blended score = 0.7 x ALS predicted_rating + 0.3 x trending_norm')
print('  5. Re-rank Top-5 per user by blended score')
print()
print('Why 70/30 split?')
print('  ALS captures stable long-term user preferences (dominant signal).')
print('  Trending captures real-time popularity shifts (secondary signal).')
print('  70/30 ensures recommendations stay personalized but react to trends.')
print()
print(f'Output saved to : /data/integrated_recs.parquet')
print(f'Latency         : {latency:.3f}s')

spark.stop()
print()
print('Spark session stopped.')

=== Integration Summary ===

Data sources:
  Batch  — ALS model          : /data/als_model
  Batch  — Pre-gen Top-5 recs : /data/user_top5_recs.parquet
  Stream — Window analytics   : /data/streaming_output/

Integration logic:
  1. Load ALS Top-5 per user (historical preference)
  2. Load latest trending score per item (real-time signal)
  3. Normalize trending score to [0, 5] (same scale as ALS rating)
  4. Blended score = 0.7 x ALS predicted_rating + 0.3 x trending_norm
  5. Re-rank Top-5 per user by blended score

Why 70/30 split?
  ALS captures stable long-term user preferences (dominant signal).
  Trending captures real-time popularity shifts (secondary signal).
  70/30 ensures recommendations stay personalized but react to trends.

Output saved to : /data/integrated_recs.parquet
Latency         : 201.053s

Spark session stopped.
